In [4]:
import os
import re
import time
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs
from pathlib import Path

In [2]:
def ensure_dir(f):
    if not os.path.exists(f):
        os.makedirs(f)

In [11]:
BASE_DIR = os.getcwd()  # current working directory
MAIN_FOLDER = "Commission DG AT"
DEST_DIR = os.path.join(BASE_DIR, MAIN_FOLDER)
ensure_dir(DEST_DIR)

# Column names (as detected)
CASE_COL      = 'Case number'
TITLE_COL     = 'Case title'
DECISIONS_COL = 'Decisions'

In [12]:
df = pd.read_csv(os.path.join(DEST_DIR, "antitrust_cartel.csv"))
print('Columns:', list(df.columns))
print('Rows:', len(df))
df.head(3)

Columns: ['Case number', 'Case title', 'Antitrust / Cartels', 'Companies', 'Last decision date', 'Economic activities', 'Legal basis', 'Initiation of proceedings', 'Decisions', 'Other case related information']
Rows: 742


,Case number,Case title,Antitrust / Cartels,Companies,Last decision date,Economic activities,Legal basis,Initiation of proceedings,Decisions,Other case related information
0,AT.28841,ABG + AVIA / Gulf + Chevron + Texaco + Mobil +...,Antitrust,Texaco Inc.\nhttps://competition-cases.ec.euro...,19.04.1977,B.06.1 - Extraction of crude petroleum (NACE R...,Art. 102 TFEU,NaN,19.04.1977 - Prohibition Decision\n Decision ...,NaN
1,AT.29629,LAMINES ET ALLIAGES DE ZINC,Cartel,NaN,14.12.1982,"C.24.43 - Lead, zinc and tin production (NACE ...",NaN,NaN,14.12.1982 - Prohibition Decision\n Decision ...,NaN
2,AT.33585,PO/RAPPORTS CHEMINS DE FER+AGENCES DE VOYAGES,Cartel,NaN,25.11.1992,H.49 - Land transport and transport via pipeli...,Art. 101 TFEU,NaN,25.11.1992 - Prohibition Decision\n Decision ...,NaN


In [13]:
# Loop through each row and create a subfolder named after the Case number
for case_no in df[CASE_COL].dropna().astype(str):
    folder_path = os.path.join(DEST_DIR, case_no.strip())
    ensure_dir(folder_path)

print(f"✅ Created {len(df[CASE_COL].dropna())} folders inside '{DEST_DIR}'")

✅ Created 742 folders inside 'C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\Commission DG AT'


In [19]:
URL_RE = re.compile(r'https?://\S+')

def extract_urls(text: str):
    if not isinstance(text, str) or not text.strip():
        return []
    urls = []
    for m in URL_RE.finditer(text):
        u = m.group(0).rstrip(').,;]}>\"\'\u00A0')
        urls.append((m.start(), m.end(), u))   # keep spans so we can name files from the surrounding text
    # de-duplicate while preserving order
    seen, out = set(), []
    for s, e, u in urls:
        if u not in seen:
            seen.add(u)
            out.append((s, e, u))
    return out

def label_before_url(text: str, start_idx: int, max_back: int = 180) -> str:
    """
    Return the text immediately before the URL (e.g., 'Decision text: EN published on 19.04.1977').
    - Looks back up to `max_back` chars.
    - Stops at the last newline if present.
    - Cleans whitespace and removes any newlines.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    # Take a window of text before the URL
    window = text[max(0, start_idx - max_back): start_idx]

    # Only keep the last line before a newline
    if "\n" in window:
        window = window.split("\n")[-1]

    label = window.strip()

    # Remove trailing punctuation or dashes
    label = re.sub(r"[-–—\s:]*$", "", label)
    # Collapse extra spaces
    label = re.sub(r"\s+", " ", label).strip()

    # Truncate for safety
    if len(label) > 120:
        label = label[-120:]

    return label

In [20]:
for i, row in df.iterrows():
    decisions_text = row.get(DECISIONS_COL, "")
    spans = extract_urls(decisions_text)
    print(f"Row {i} — Case {row.get(CASE_COL,'').strip()}: {len(spans)} URL(s)")
    for j, (s_idx, _, u) in enumerate(spans, 1):
        label = label_before_url(decisions_text, s_idx)
        if label:
            print(f"   {j}) {label} → {u}")
        else:
            print(f"   {j}) {u}")
    print("-" * 80)


Row 0 — Case AT.28841: 1 URL(s)
   1) Decision text: EN published on 19.04.1977 → https://ec.europa.eu/competition/antitrust/cases/dec_docs/28841/28841_2_7.pdf
--------------------------------------------------------------------------------
Row 1 — Case AT.29629: 1 URL(s)
   1) Decision text: EN published on 14.12.1982 → https://ec.europa.eu/competition/antitrust/cases/dec_docs/29629/29629_3_6.pdf
--------------------------------------------------------------------------------
Row 2 — Case AT.33585: 1 URL(s)
   1) Decision text: EN published on 25.11.1992 → https://ec.europa.eu/competition/antitrust/cases/dec_docs/33585/33585_2_4.pdf
--------------------------------------------------------------------------------
Row 3 — Case AT.32948: 1 URL(s)
   1) Publication in the OJ: OJEU L/1994/378 page 45 of 21.12.1994 → https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=uriserv:OJ.L_.1994.378.01.0045.01.ENG&toc=OJ:L:1994:378:TOC
----------------------------------------------------------------

In [22]:
# --- Minimal patch (uses your existing DEST_DIR) ---

from pathlib import Path
import csv, time, os, re, hashlib
from urllib.parse import urlparse, unquote
import requests

# 1) Normalize DEST_DIR (whatever you already set) and ensure it exists
if not isinstance(DEST_DIR, Path):
    DEST_DIR = Path(DEST_DIR)
DEST_DIR.mkdir(parents=True, exist_ok=True)

# 2) Manifest path
MANIFEST = DEST_DIR / "manifest.csv"

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def sanitize(s: str, max_len: int = 150) -> str:
    s = (s or "").strip().replace("\n", " ").replace("\r", " ")
    s = re.sub(r'[\\/:*?"<>|]+', "_", s)
    s = re.sub(r"\s+", " ", s).strip()
    return (s[:max_len].rstrip() or "document")

def extension_from_url(url: str) -> str:
    tail = Path(unquote(urlparse(url).path)).name
    return ("." + tail.split(".")[-1].lower()) if "." in tail else ""

def guess_filename_from_url(url: str) -> str:
    tail = Path(unquote(urlparse(url).path)).name
    if tail:
        return sanitize(tail, 150)
    h = hashlib.sha1(url.encode("utf-8")).hexdigest()[:10]
    return f"document_{h}.html"

def ensure_unique_path(dest_dir: Path, filename: str) -> Path:
    base = dest_dir / filename
    if not base.exists(): return base
    stem, ext = os.path.splitext(filename)
    k = 1
    while True:
        cand = dest_dir / f"{stem}_{k}{ext}"
        if not cand.exists(): return cand
        k += 1

def safe_relpath(p: Path, base: Path) -> str:
    try:
        return str(p.relative_to(base))
    except Exception:
        return str(p)

# Simple downloader
DRY_RUN = False  # set True to test without writing files

def download(url: str, dest: Path, timeout: int = 30, retries: int = 2):
    if DRY_RUN:
        return "ok", 0, 0, ""
    last_err = ""
    for attempt in range(retries + 1):
        try:
            with requests.get(
                url,
                headers={"User-Agent": "Mozilla/5.0 (compatible; EC-Downloader/1.0)"},
                timeout=timeout,
                stream=True,
            ) as r:
                code = r.status_code
                r.raise_for_status()
                ensure_dir(dest.parent)
                nbytes = 0
                with open(dest, "wb") as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk); nbytes += len(chunk)
            return "ok", code, nbytes, ""
        except Exception as e:
            last_err = str(e)
            time.sleep(0.6 * (attempt + 1))
    return "error", -1, 0, last_err

def log_manifest(row: dict):
    header = ["timestamp","case_number","url","dest_path","status","http_code","bytes","error"]
    is_new = not MANIFEST.exists()
    ensure_dir(MANIFEST.parent)
    with open(MANIFEST, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=header)
        if is_new: w.writeheader()
        w.writerow(row)

In [24]:
# --- PATCH: Rewrite legacy RAPID ".do" links to modern Presscorner PDFs ---

import re
from urllib.parse import urlparse, parse_qs

def rewrite_press_do_url(url: str, prefer_lang: str = "en") -> str | None:
    """
    If `url` is a RAPID .do press link like
    http://europa.eu/rapid/pressReleasesAction.do?reference=IP_97_405
    return the modern Presscorner print-PDF link:
    https://ec.europa.eu/commission/presscorner/api/files/document/print/en/ip_97_405/IP_97_405_EN.pdf
    Otherwise return None.
    """
    try:
        u = urlparse(url)
        path = (u.path or "").lower()
        host = (u.netloc or "").lower()

        # Only handle europa.eu RAPID .do links
        if ".do" not in path or "europa.eu" not in host:
            return None
        if not re.search(r"(pressreleasesaction|pressrelease)", path):
            return None

        qs = parse_qs(u.query or "")
        ref = ""
        if "reference" in qs and qs["reference"]:
            ref = qs["reference"][0]
        else:
            m = re.search(r"reference=([A-Za-z]+[-_]\d{2}[-_]\d+)", url, flags=re.I)
            ref = m.group(1) if m else ""

        if not ref:
            return None

        ref_clean = ref.strip().replace("-", "_")
        ref_upper = ref_clean.upper()
        ref_lower = ref_upper.lower()
        lang = (prefer_lang or "en").lower()

        return (
            f"https://ec.europa.eu/commission/presscorner/api/files/document/print/"
            f"{lang}/{ref_lower}/{ref_upper}_{lang.upper()}.pdf"
        )
    except Exception:
        return None


def maybe_rewrite_legacy_do(url: str, prefer_lang: str = "en") -> str:
    """
    Return rewritten Presscorner URL if it's a legacy RAPID .do press link,
    otherwise return the original.
    """
    newu = rewrite_press_do_url(url, prefer_lang=prefer_lang)
    return newu if newu else url


In [25]:
# === MAIN LOOP ===
for i, row in df.iterrows():
    case_no = str(row.get(CASE_COL, "")).strip()
    if not case_no:
        continue

    decisions_text = row.get(DECISIONS_COL, "")
    spans = extract_urls(decisions_text)
    if not spans:
        continue

    case_dir = DEST_DIR / sanitize(case_no)
    ensure_dir(case_dir)

    print(f"[{i}] Case {case_no}: {len(spans)} URL(s)")
    for j, (start_idx, _end_idx, url) in enumerate(spans, 1):

        # --- ADDENDUM 1: rewrite legacy .do press links ---
        orig_url = url
        url = maybe_rewrite_legacy_do(url, prefer_lang="en")

        # --- Build readable filename from text before the URL ---
        label = label_before_url(decisions_text, start_idx)
        label = sanitize(label or os.path.splitext(guess_filename_from_url(url))[0], 120)

        # --- ADDENDUM 2: detect press releases for naming clarity ---
        if url != orig_url and "presscorner/api/files/document/print" in url.lower():
            if not re.search(r"\bpress\b|\bcommunication\b|\brelease\b", label, flags=re.I):
                label = f"Press communication - {label}"

        ext = extension_from_url(url)
        if not ext:
            ext = ".pdf" if url.lower().endswith(".pdf") else ".html"

        # --- ADDENDUM 3: force .pdf extension for rewritten presscorner URLs ---
        if url != orig_url:
            ext = ".pdf"

        filename = f"{j:02d} - {label}{ext}"
        dest = ensure_unique_path(case_dir, filename)

        # --- Download ---
        status, code, nbytes, err = download(url, dest)
        print(f"   {j:02d}) {status:>5} | {code:>3} | {nbytes:>8} B | {dest.name}")
        if status != "ok" and err:
            print(f"        ↳ error: {err[:200]}")

        # --- Log progress ---
        log_manifest({
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "case_number": case_no,
            "url": url,
            "dest_path": safe_relpath(dest, DEST_DIR),
            "status": status,
            "http_code": code,
            "bytes": nbytes,
            "error": err,
        })
    print("-" * 80)

[0] Case AT.28841: 1 URL(s)
   01)    ok | 200 |  1989768 B | 01 - Decision text_ EN published on 19.04.1977_1.pdf
--------------------------------------------------------------------------------
[1] Case AT.29629: 1 URL(s)
   01)    ok | 200 |   166191 B | 01 - Decision text_ EN published on 14.12.1982_1.pdf
--------------------------------------------------------------------------------
[2] Case AT.33585: 1 URL(s)
   01)    ok | 200 |   136879 B | 01 - Decision text_ EN published on 25.11.1992_1.pdf
--------------------------------------------------------------------------------
[3] Case AT.32948: 1 URL(s)
   01)    ok | 200 |   225458 B | 01 - Publication in the OJ_ OJEU L_1994_378 page 45 of 21.12.1994_1.html
--------------------------------------------------------------------------------
[4] Case AT.35388: 1 URL(s)
   01)    ok | 200 |  1708062 B | 01 - Decision text_ FR published on 06.07.2012_1.pdf
--------------------------------------------------------------------------------
